# 06 — Final Explainable Risk Engine
## Historical-Signal-Based Yield Risk Score

This notebook implements the final MVP risk engine selected after the machine
learning experiments.

The output is an explainable **0–100 yield-risk score** based on historical
signals.

The score is not presented as a guaranteed future yield forecast.

Instead, it summarizes adverse historical conditions that may indicate elevated
yield risk.

---

## Final Drivers

The model uses four lagged historical signals:

1. previous yield trend,
2. previous rainfall anomaly,
3. previous cultivated-area trend,
4. previous price volatility.

Each component is scaled to 0–100 and combined using transparent weights.

---

## Base Weights

- Yield trend: **35%**
- Rainfall anomaly: **30%**
- Area trend: **20%**
- Price volatility: **15%**

Missing drivers are excluded and the remaining available weights are
re-normalized.

Missing data is never treated as zero risk.

# 1. Import Libraries and Load the Analytical Master

In [1]:
import pandas as pd
import numpy as np
import json

from pathlib import Path

PROCESSED_DIR = Path("../../data/processed")

analytics_master = pd.read_csv(
    PROCESSED_DIR
    / "analytics_master_2021_2025.csv"
)

print("Rows:", len(analytics_master))

Rows: 1403


# 2. Sort the Historical Series

All lagged calculations must happen in chronological order within:

`State + District + Crop`

Sorting first protects against accidentally shifting values from the wrong
period.

In [2]:
GROUP_KEYS = [
    "State",
    "District",
    "Crop"
]

analytics_master = (
    analytics_master
    .sort_values(
        GROUP_KEYS + ["Year"]
    )
    .reset_index(drop=True)
)

# 3. Recreate Yield and Area Change Features

If the current integrated file does not already contain annual percentage
changes, they are calculated directly from:

- `Yield`
- `Area`

within each State-District-Crop history.

These values describe how the observed variable changed relative to the
previous crop year.

The first year in each history naturally has no percentage change.

In [3]:
if "Yield_Change_Pct" not in analytics_master.columns:
    analytics_master[
        "Yield_Change_Pct"
    ] = (
        analytics_master
        .groupby(
            GROUP_KEYS
        )[
            "Yield"
        ]
        .pct_change()
        * 100
    )

if "Area_Change_Pct" not in analytics_master.columns:
    analytics_master[
        "Area_Change_Pct"
    ] = (
        analytics_master
        .groupby(
            GROUP_KEYS
        )[
            "Area"
        ]
        .pct_change()
        * 100
    )

# Percentage change can become +/- infinity if the previous denominator is 0.
# Infinite values are not valid risk signals, so they are converted to NaN.
analytics_master[
    [
        "Yield_Change_Pct",
        "Area_Change_Pct"
    ]
] = analytics_master[
    [
        "Yield_Change_Pct",
        "Area_Change_Pct"
    ]
].replace(
    [np.inf, -np.inf],
    np.nan
)

# 4. Create Lagged Risk Signals

The risk score for a row uses **previous-period evidence**.

For example, the 2024-25 score uses historical signals carried from the prior
period rather than using 2024-25 yield performance to score itself.

Lagged variables:

- `Prev_Yield_Change_Pct`
- `Prev_Area_Change_Pct`
- `Prev_Rainfall_Deviation_Pct`
- `Prev_Price_Volatility_Pct`

In [4]:
analytics_master[
    "Prev_Yield_Change_Pct"
] = analytics_master.groupby(
    GROUP_KEYS
)[
    "Yield_Change_Pct"
].shift(1)

analytics_master[
    "Prev_Area_Change_Pct"
] = analytics_master.groupby(
    GROUP_KEYS
)[
    "Area_Change_Pct"
].shift(1)

analytics_master[
    "Prev_Rainfall_Deviation_Pct"
] = analytics_master.groupby(
    GROUP_KEYS
)[
    "Rainfall_Deviation_Pct"
].shift(1)

analytics_master[
    "Prev_Price_Volatility_Pct"
] = analytics_master.groupby(
    GROUP_KEYS
)[
    "Price_Volatility_Pct"
].shift(1)

# 5. Missingness of Latest-Year Drivers

The 2024-25 score cannot assume that all four historical drivers exist.

The verified run contained 355 State-District-Crop rows for 2024-25.

Availability before scoring was approximately:

- previous yield-change signal: 348 / 355
- previous area-change signal: 310 / 355
- previous rainfall-deviation signal: 350 / 355
- previous price-volatility signal: 133 / 355

Price volatility is therefore the sparsest driver.

The risk engine must be able to operate without treating a missing driver as a
safe/zero-risk driver.

In [5]:
latest = analytics_master[
    analytics_master[
        "Year"
    ] == "2024-25"
].copy()

driver_columns = [
    "Prev_Yield_Change_Pct",
    "Prev_Area_Change_Pct",
    "Prev_Rainfall_Deviation_Pct",
    "Prev_Price_Volatility_Pct"
]

for column in driver_columns:
    print(
        column,
        "available:",
        latest[column].notna().sum(),
        "/",
        len(latest)
    )

Prev_Yield_Change_Pct available: 348 / 355
Prev_Area_Change_Pct available: 303 / 355
Prev_Rainfall_Deviation_Pct available: 350 / 355
Prev_Price_Volatility_Pct available: 133 / 355


# 6. Convert Historical Signals into Adverse Signals

Each driver must be converted into a value where:

**larger = more adverse**

## Yield

Only a decline is considered adverse.

`Yield_Adverse_Signal = max(0, -Prev_Yield_Change_Pct)`

A previous yield increase does not add downside risk.

## Area

Only a decline in cultivated area is considered adverse.

`Area_Adverse_Signal = max(0, -Prev_Area_Change_Pct)`

## Rainfall

Both unusually low and unusually high rainfall can create agricultural stress.

Therefore rainfall uses the absolute deviation:

`abs(Prev_Rainfall_Deviation_Pct)`

## Price

Higher observed price volatility contributes a larger adverse signal.

In [6]:
analytics_master[
    "Yield_Adverse_Signal"
] = (
    -analytics_master[
        "Prev_Yield_Change_Pct"
    ]
).clip(lower=0)

analytics_master[
    "Area_Adverse_Signal"
] = (
    -analytics_master[
        "Prev_Area_Change_Pct"
    ]
).clip(lower=0)

analytics_master[
    "Rainfall_Adverse_Signal"
] = analytics_master[
    "Prev_Rainfall_Deviation_Pct"
].abs()

analytics_master[
    "Price_Adverse_Signal"
] = analytics_master[
    "Prev_Price_Volatility_Pct"
].clip(lower=0)

# 7. Learn Scaling References from Historical Years Only

Raw drivers use different units.

For example:

- yield change is a percentage,
- rainfall anomaly is a percentage,
- price volatility is a percentage.

To combine them, each adverse signal is scaled to a 0–100 component.

The reference value is the **95th percentile** of the adverse signal calculated
from historical scaling years:

- 2022-23
- 2023-24

Using a percentile rather than the absolute maximum reduces sensitivity to a
single extreme outlier.

The latest 2024-25 period is not used to learn the scaling reference.

In [7]:
SCALING_YEARS = [
    "2022-23",
    "2023-24"
]

historical_scaling = analytics_master[
    analytics_master[
        "Year"
    ].isin(
        SCALING_YEARS
    )
].copy()

REFERENCE_VALUES = {
    "Yield": historical_scaling[
        "Yield_Adverse_Signal"
    ].quantile(0.95),

    "Area": historical_scaling[
        "Area_Adverse_Signal"
    ].quantile(0.95),

    "Rainfall": historical_scaling[
        "Rainfall_Adverse_Signal"
    ].quantile(0.95),

    "Price": historical_scaling[
        "Price_Adverse_Signal"
    ].quantile(0.95)
}

REFERENCE_VALUES

{'Yield': np.float64(21.417356499831723),
 'Area': np.float64(9.371366279069777),
 'Rainfall': np.float64(35.16355942791774),
 'Price': np.float64(12.303935433226039)}

## Verified 95th-Percentile References

The completed project run produced approximately:

```text
Yield adverse reference:     21.42
Area adverse reference:       9.37
Rainfall adverse reference:  35.16
Price adverse reference:     12.30
```

These values are data-derived thresholds, not manually chosen risk cutoffs.

# 8. Scale Every Driver to 0–100

For each component:

`Component Score = adverse signal / historical reference × 100`

The result is clipped to:

`0 ≤ component ≤ 100`

Therefore a signal at or above the historical 95th-percentile adverse level
receives approximately the maximum component score.

In [8]:
def scale_component(
    series,
    reference_value
):
    if pd.isna(reference_value) or reference_value <= 0:
        return pd.Series(
            np.nan,
            index=series.index
        )

    return (
        series
        / reference_value
        * 100
    ).clip(
        lower=0,
        upper=100
    )


analytics_master[
    "Yield_Risk_Component"
] = scale_component(
    analytics_master[
        "Yield_Adverse_Signal"
    ],
    REFERENCE_VALUES["Yield"]
)

analytics_master[
    "Area_Risk_Component"
] = scale_component(
    analytics_master[
        "Area_Adverse_Signal"
    ],
    REFERENCE_VALUES["Area"]
)

analytics_master[
    "Rainfall_Risk_Component"
] = scale_component(
    analytics_master[
        "Rainfall_Adverse_Signal"
    ],
    REFERENCE_VALUES["Rainfall"]
)

analytics_master[
    "Price_Risk_Component"
] = scale_component(
    analytics_master[
        "Price_Adverse_Signal"
    ],
    REFERENCE_VALUES["Price"]
)

# 9. Base Risk Weights

The final base weights are:

| Driver | Weight |
|---|---:|
| Yield trend | 0.35 |
| Rainfall anomaly | 0.30 |
| Area trend | 0.20 |
| Price volatility | 0.15 |

The weights sum to 1.0 when all four drivers are present.

However, missing drivers are not assigned component score 0.

Instead, only available component weights participate in the denominator.

In [9]:
RISK_WEIGHTS = {
    "Yield_Risk_Component": 0.35,
    "Rainfall_Risk_Component": 0.30,
    "Area_Risk_Component": 0.20,
    "Price_Risk_Component": 0.15
}

# 10. Dynamic Weight Re-Normalization

Suppose price volatility is missing.

Incorrect approach:

`missing price component = 0`

That would incorrectly imply the observed price signal was perfectly safe.

Correct approach:

1. remove the missing component,
2. sum only available weights,
3. divide available weighted contributions by available weight.

Conceptually:

`Risk = Σ(component × weight) / Σ(available weight)`

This keeps missingness separate from actual low risk.

In [10]:
# Individual weighted contributions.
for component, weight in RISK_WEIGHTS.items():
    contribution_name = (
        component.replace(
            "_Risk_Component",
            "_Weighted_Contribution"
        )
    )

    analytics_master[
        contribution_name
    ] = (
        analytics_master[
            component
        ]
        * weight
    )

# Available driver weight for each row.
analytics_master[
    "Available_Risk_Weight"
] = 0.0

for component, weight in RISK_WEIGHTS.items():
    analytics_master[
        "Available_Risk_Weight"
    ] += np.where(
        analytics_master[
            component
        ].notna(),
        weight,
        0.0
    )

weighted_columns = [
    "Yield_Weighted_Contribution",
    "Rainfall_Weighted_Contribution",
    "Area_Weighted_Contribution",
    "Price_Weighted_Contribution"
]

weighted_sum = (
    analytics_master[
        weighted_columns
    ]
    .fillna(0)
    .sum(axis=1)
)

analytics_master[
    "Risk_Score"
] = np.where(
    analytics_master[
        "Available_Risk_Weight"
    ] > 0,
    (
        weighted_sum
        / analytics_master[
            "Available_Risk_Weight"
        ]
    ),
    np.nan
)

analytics_master[
    "Risk_Score"
] = analytics_master[
    "Risk_Score"
].clip(
    lower=0,
    upper=100
)

# 11. Count Available Drivers

The number of non-missing risk components is stored as:

`Drivers_Available`

This lets the dashboard communicate how much evidence supported the score.

The verified 2024-25 run produced:

```text
0 drivers →   5 rows
1 driver  →   2 rows
2 drivers →  40 rows
3 drivers → 180 rows
4 drivers → 128 rows
```

In [11]:
component_columns = list(
    RISK_WEIGHTS.keys()
)

analytics_master[
    "Drivers_Available"
] = analytics_master[
    component_columns
].notna().sum(axis=1)

# 12. Risk Data Quality

Data quality is based on the number of available drivers.

| Drivers | Quality |
|---:|---|
| 4 | Complete |
| 3 | Good |
| 2 | Limited |
| 1 | Very Limited |
| 0 | Insufficient |

The label describes **input coverage**, not whether a score is high or low.

In [12]:
def risk_data_quality(
    drivers_available
):
    if drivers_available == 4:
        return "Complete"
    elif drivers_available == 3:
        return "Good"
    elif drivers_available == 2:
        return "Limited"
    elif drivers_available == 1:
        return "Very Limited"
    else:
        return "Insufficient"


analytics_master[
    "Risk_Data_Quality"
] = analytics_master[
    "Drivers_Available"
].apply(
    risk_data_quality
)

# 13. Dashboard Publication Rule

The model can mathematically produce a score from only one available component.

However, a one-driver score is too weak to publish as the main dashboard risk
indicator.

Therefore:

`Dashboard_Risk_Score`

is available only when:

`Drivers_Available >= 2`

Rows with fewer than two drivers remain in the dataset, but their published
dashboard score is missing.

In [13]:
analytics_master[
    "Dashboard_Risk_Score"
] = analytics_master[
    "Risk_Score"
].where(
    analytics_master[
        "Drivers_Available"
    ] >= 2
)

# 14. Extract the Top Contributing Drivers

The dashboard should explain **why** a row received its score.

Drivers are ranked using their weighted contribution.

Only drivers whose contribution is:

- available,
- and greater than zero

are eligible.

This prevents a driver with zero contribution from being presented as a
meaningful risk reason.

In [14]:
DRIVER_LABELS = {
    "Yield_Weighted_Contribution": "Yield trend",
    "Rainfall_Weighted_Contribution": "Rainfall anomaly",
    "Area_Weighted_Contribution": "Area trend",
    "Price_Weighted_Contribution": "Price volatility"
}


def get_top_drivers(row):
    contributions = []

    for column, label in DRIVER_LABELS.items():
        value = row[column]

        if (
            pd.notna(value)
            and value > 0
        ):
            contributions.append(
                (label, value)
            )

    contributions.sort(
        key=lambda item: item[1],
        reverse=True
    )

    labels = [
        item[0]
        for item in contributions[:3]
    ]

    while len(labels) < 3:
        labels.append(None)

    return pd.Series(
        labels,
        index=[
            "Top_Driver_1",
            "Top_Driver_2",
            "Top_Driver_3"
        ]
    )


analytics_master[
    [
        "Top_Driver_1",
        "Top_Driver_2",
        "Top_Driver_3"
    ]
] = analytics_master.apply(
    get_top_drivers,
    axis=1
)

# 15. Validate the 2024-25 Risk Output

The verified project output contains:

- **355 rows** for 2024-25
- **348 usable dashboard risk scores**
- **7 unavailable dashboard risk scores**

The 7 unavailable rows have fewer than two usable drivers.

The raw normalized risk distribution for rows with sufficient inputs was
approximately:

- mean: 34.80
- median: 34.14
- minimum: 0.76
- maximum: 100

In [15]:
latest = analytics_master[
    analytics_master[
        "Year"
    ] == "2024-25"
].copy()

print(
    "2024-25 rows:",
    len(latest)
)

print(
    "Usable dashboard risk:",
    latest[
        "Dashboard_Risk_Score"
    ].notna().sum()
)

print(
    "Unavailable dashboard risk:",
    latest[
        "Dashboard_Risk_Score"
    ].isna().sum()
)

print("\nDrivers available:")
print(
    latest[
        "Drivers_Available"
    ].value_counts()
    .sort_index()
)

print("\nRisk score summary:")
print(
    latest[
        "Risk_Score"
    ].describe()
)

2024-25 rows: 355
Usable dashboard risk: 348
Unavailable dashboard risk: 7

Drivers available:
Drivers_Available
0      5
1      2
2     40
3    180
4    128
Name: count, dtype: int64

Risk score summary:
count    350.000000
mean      34.803442
std       17.489677
min        0.760574
25%       21.898911
50%       34.137200
75%       44.500948
max      100.000000
Name: Risk_Score, dtype: float64


# 16. Inspect Highest-Risk Examples

High scores are useful for sanity checking whether the component logic and top
drivers agree.

The score should not be interpreted as a probability of crop failure.

It is a normalized historical-signal risk index.

In [16]:
display_columns = [
    "State",
    "District",
    "Crop",
    "Year",
    "Dashboard_Risk_Score",
    "Risk_Data_Quality",
    "Drivers_Available",
    "Top_Driver_1",
    "Top_Driver_2",
    "Top_Driver_3",
    "Yield_Risk_Component",
    "Rainfall_Risk_Component",
    "Area_Risk_Component",
    "Price_Risk_Component"
]

latest[
    display_columns
].sort_values(
    "Dashboard_Risk_Score",
    ascending=False
).head(20)

,State,District,Crop,Year,Dashboard_Risk_Score,Risk_Data_Quality,Drivers_Available,Top_Driver_1,Top_Driver_2,Top_Driver_3,Yield_Risk_Component,Rainfall_Risk_Component,Area_Risk_Component,Price_Risk_Component
62,Haryana,Gurugram,Rice,2024-25,97.743947,Good,3,Yield trend,Rainfall anomaly,Area trend,94.521014,100.000000,100.000000,NaN
820,Uttar Pradesh,Fatehpur,Maize,2024-25,91.978630,Limited,2,Yield trend,Rainfall anomaly,None,100.000000,82.620365,NaN,NaN
1146,Uttar Pradesh,Mau,Rice,2024-25,89.339529,Good,3,Yield trend,Rainfall anomaly,Area trend,97.470872,72.745981,100.000000,NaN
1034,Uttar Pradesh,Kaushambi,Maize,2024-25,81.249541,Limited,2,Yield trend,Rainfall anomaly,None,100.000000,59.374006,NaN,NaN
1226,Uttar Pradesh,Prayagraj,Maize,2024-25,80.302814,Limited,2,Yield trend,Rainfall anomaly,None,100.000000,57.322764,NaN,NaN
574,Uttar Pradesh,Auraiya,Rice,2024-25,80.266290,Good,3,Yield trend,Rainfall anomaly,Area trend,73.337472,75.194105,100.000000,NaN
748,Uttar Pradesh,Chandauli,Maize,2024-25,79.150924,Limited,2,Yield trend,Rainfall anomaly,None,100.000000,54.827003,NaN,NaN
405,Punjab,Moga,Maize,2024-25,77.840529,Limited,2,Yield trend,Rainfall anomaly,None,100.000000,51.987813,NaN,NaN
1166,Uttar Pradesh,Mirzapur,Maize,2024-25,76.482629,Good,3,Yield trend,Area trend,Rainfall anomaly,100.000000,33.367449,100.000000,NaN
235,Haryana,Yamunanagar,Maize,2024-25,75.822705,Limited,2,Yield trend,Rainfall anomaly,None,100.000000,47.615862,NaN,NaN


# 17. Save the Full Analytical Risk Dataset

The full historical analytical table including risk components is saved as:

`analytics_master_with_risk_2021_2025.csv`

This file is useful for:

- auditing,
- historical analysis,
- debugging,
- and future model development.

In [17]:
analytics_master.to_csv(
    PROCESSED_DIR
    / "analytics_master_with_risk_2021_2025.csv",
    index=False
)

print(
    "Saved analytics_master_with_risk_2021_2025.csv"
)

Saved analytics_master_with_risk_2021_2025.csv


# 18. Save the 2024-25 Dashboard Risk Output

The dashboard does not need every intermediate engineering column.

A smaller latest-year output is therefore created with:

- identifying fields,
- published risk score,
- data quality,
- top drivers,
- component scores,
- selected historical values used for trends/context.

In [18]:
dashboard_columns = [
    "State",
    "District",
    "Crop",
    "Year",
    "Dashboard_Risk_Score",
    "Risk_Data_Quality",
    "Drivers_Available",
    "Top_Driver_1",
    "Top_Driver_2",
    "Top_Driver_3",
    "Yield_Risk_Component",
    "Rainfall_Risk_Component",
    "Area_Risk_Component",
    "Price_Risk_Component",
    "Avg_Modal_Price",
    "Crop_Year_Rainfall_mm",
    "Area",
    "Yield"
]

dashboard_output = latest[
    dashboard_columns
].copy()

dashboard_output.to_csv(
    PROCESSED_DIR
    / "dashboard_risk_output_2024_25.csv",
    index=False
)

print(
    "Saved dashboard_risk_output_2024_25.csv"
)

Saved dashboard_risk_output_2024_25.csv


# 19. Save Risk-Model Configuration

The model configuration is saved separately so the scoring methodology can be
audited without reading Python code.

The configuration records:

- model type,
- driver weights,
- historical scaling method,
- scaling years,
- reference thresholds,
- score range,
- minimum drivers,
- missing-data handling.

In [19]:
risk_model_config = {
    "model_type": (
        "Explainable Weighted Rule-Based Risk Model"
    ),
    "interpretation": (
        "Historical-signal-based yield risk assessment"
    ),
    "weights": {
        "yield_trend": 0.35,
        "rainfall_anomaly": 0.30,
        "area_trend": 0.20,
        "price_volatility": 0.15
    },
    "scaling_method": (
        "95th percentile of historical adverse signals"
    ),
    "scaling_years": SCALING_YEARS,
    "reference_values": {
        key: (
            None
            if pd.isna(value)
            else float(value)
        )
        for key, value in REFERENCE_VALUES.items()
    },
    "score_range": [0, 100],
    "minimum_dashboard_drivers": 2,
    "missing_driver_handling": (
        "Exclude missing driver and re-normalize "
        "remaining available weights"
    )
}

with open(
    PROCESSED_DIR
    / "risk_model_config.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        risk_model_config,
        file,
        indent=2
    )

print("Saved risk_model_config.json")

Saved risk_model_config.json


# 20. Verify the Meaning of the Source Yield Field

Before handing data to the backend, the production source's `Yield` field must
not be confused with total production.

The relationship was tested against:

`Production × 1000 / Area`

After removing rows where `Area <= 0`, the verified results were:

- median relative error: **0.296%**
- mean relative error: **2.585%**
- rows within 1%: **931 / 1228**
- rows within 5%: **1093 / 1228**

This provides strong numerical evidence that the source Yield field behaves
like a **yield rate**, not total tonnes.

Under the project unit interpretation, the backend field is represented as:

`yield_kg_per_hectare`

Production remains a separate field.

In [20]:
unit_check = (
    analytics_master[
        [
            "State",
            "District",
            "Crop",
            "Year",
            "Area",
            "Production",
            "Yield"
        ]
    ]
    .dropna()
    .copy()
)

# Division by zero would produce infinity, so only positive-area rows can be
# used for this unit consistency check.
unit_check = unit_check[
    unit_check[
        "Area"
    ] > 0
].copy()

unit_check[
    "Calculated_Yield"
] = (
    unit_check[
        "Production"
    ]
    * 1000
    / unit_check[
        "Area"
    ]
)

unit_check[
    "Yield_Relative_Error_Pct"
] = (
    (
        unit_check[
            "Yield"
        ]
        - unit_check[
            "Calculated_Yield"
        ]
    ).abs()
    / unit_check[
        "Yield"
    ].abs()
    * 100
)

print(
    "Median relative error (%):",
    unit_check[
        "Yield_Relative_Error_Pct"
    ].median()
)

print(
    "Mean relative error (%):",
    unit_check[
        "Yield_Relative_Error_Pct"
    ].mean()
)

print(
    "Rows within 1%:",
    (
        unit_check[
            "Yield_Relative_Error_Pct"
        ] <= 1
    ).sum()
)

print(
    "Rows within 5%:",
    (
        unit_check[
            "Yield_Relative_Error_Pct"
        ] <= 5
    ).sum()
)

print(
    "Total checked rows:",
    len(unit_check)
)

Median relative error (%): 0.2959070166667285
Mean relative error (%): 2.585062510402853
Rows within 1%: 931
Rows within 5%: 1093
Total checked rows: 1228


# 21. Create the Backend Handoff Dataset

The backend should receive a clean, compact dataset rather than the full
research table.

Final fields:

- state
- district
- crop
- year
- rainfall_mm
- avg_price
- area_hectares
- production
- yield_kg_per_hectare
- risk_score
- risk_data_quality
- drivers_available
- top_driver_1
- top_driver_2
- top_driver_3

Important:

- `year` is a string such as `2024-25`,
- missing price remains null,
- missing risk remains null,
- null risk does not mean zero risk,
- grain remains state + district + crop + year.

In [21]:
backend_handoff = analytics_master[
    [
        "State",
        "District",
        "Crop",
        "Year",
        "Crop_Year_Rainfall_mm",
        "Avg_Modal_Price",
        "Area",
        "Production",
        "Yield",
        "Dashboard_Risk_Score",
        "Risk_Data_Quality",
        "Drivers_Available",
        "Top_Driver_1",
        "Top_Driver_2",
        "Top_Driver_3"
    ]
].copy()

backend_handoff = backend_handoff.rename(
    columns={
        "State": "state",
        "District": "district",
        "Crop": "crop",
        "Year": "year",
        "Crop_Year_Rainfall_mm": "rainfall_mm",
        "Avg_Modal_Price": "avg_price",
        "Area": "area_hectares",
        "Production": "production",
        "Yield": "yield_kg_per_hectare",
        "Dashboard_Risk_Score": "risk_score",
        "Risk_Data_Quality": "risk_data_quality",
        "Drivers_Available": "drivers_available",
        "Top_Driver_1": "top_driver_1",
        "Top_Driver_2": "top_driver_2",
        "Top_Driver_3": "top_driver_3"
    }
)

# 22. Validate Backend Handoff

The completed project handoff produced:

```text
Rows: 1403
Duplicate keys: 0
Missing risk scores: 716
Missing prices: 418
Missing rainfall: 0

2024-25 rows: 355
2024-25 usable risk scores: 348
2024-25 unavailable risk scores: 7
```

The 716 historical missing risk scores are expected because early rows do not
always have sufficient lagged history.

The latest dashboard year has usable risk for 348 of 355 rows.

In [22]:
backend_keys = [
    "state",
    "district",
    "crop",
    "year"
]

print(
    "Rows:",
    len(backend_handoff)
)

print(
    "Duplicate keys:",
    backend_handoff
    .duplicated(
        subset=backend_keys
    )
    .sum()
)

print(
    "Missing risk scores:",
    backend_handoff[
        "risk_score"
    ].isna().sum()
)

print(
    "Missing prices:",
    backend_handoff[
        "avg_price"
    ].isna().sum()
)

print(
    "Missing rainfall:",
    backend_handoff[
        "rainfall_mm"
    ].isna().sum()
)

latest_backend_handoff = (
    backend_handoff[
        backend_handoff[
            "year"
        ] == "2024-25"
    ]
    .copy()
)

print(
    "2024-25 rows:",
    len(latest_backend_handoff)
)

print(
    "2024-25 usable risk scores:",
    latest_backend_handoff[
        "risk_score"
    ].notna().sum()
)

print(
    "2024-25 unavailable risk scores:",
    latest_backend_handoff[
        "risk_score"
    ].isna().sum()
)

Rows: 1403
Duplicate keys: 0
Missing risk scores: 716
Missing prices: 418
Missing rainfall: 0
2024-25 rows: 355
2024-25 usable risk scores: 348
2024-25 unavailable risk scores: 7


# 23. Save Backend Handoff Files

In [23]:
backend_handoff.to_csv(
    PROCESSED_DIR
    / "backend_handoff_2021_2025.csv",
    index=False
)

latest_backend_handoff.to_csv(
    PROCESSED_DIR
    / "backend_handoff_2024_25.csv",
    index=False
)

print(
    "Backend handoff files saved."
)

Backend handoff files saved.


# 24. Backend Data Dictionary

| Column | Meaning | Nullable? |
|---|---|---|
| `state` | State name | No |
| `district` | District name | No |
| `crop` | Wheat, Rice or Maize | No |
| `year` | Crop year such as 2024-25 | No |
| `rainfall_mm` | Crop-year rainfall | Prefer nullable |
| `avg_price` | Average available AGMARKNET modal price | Yes |
| `area_hectares` | Cultivated area | Yes |
| `production` | Source production quantity | Yes |
| `yield_kg_per_hectare` | Yield rate | Yes |
| `risk_score` | Published 0–100 dashboard risk | Yes |
| `risk_data_quality` | Input coverage label | No |
| `drivers_available` | Number of usable drivers, 0–4 | No |
| `top_driver_1` | Largest positive risk contributor | Yes |
| `top_driver_2` | Second contributor | Yes |
| `top_driver_3` | Third contributor | Yes |

### Important Null Semantics

`avg_price = null`

means AGMARKNET data was unavailable.

It does **not** mean price = 0.

`risk_score = null`

means insufficient historical evidence for the publication rule.

It does **not** mean risk = 0.

# 25. Final Model Interpretation

The final score should be described as:

> **An explainable historical-signal-based yield-risk assessment.**

It should **not** be described as:

- a guaranteed crop-failure probability,
- an exact future yield forecast,
- or a fully trained production forecasting model.

The score summarizes adverse historical patterns in:

- yield,
- rainfall,
- cultivated area,
- and market-price volatility.

This framing matches what the available data can responsibly support.

# 26. Complete Final Pipeline

```text
Production
   +
Rainfall
   +
AGMARKNET Price
   ↓
Integrated analytical master
   ↓
Create historical lag features
   ↓
Convert each signal to adverse direction
   ↓
Learn 95th-percentile historical scaling
   ↓
Convert drivers to 0–100 component scores
   ↓
Apply base weights
   ↓
Exclude missing drivers
   ↓
Re-normalize available weights
   ↓
0–100 Risk_Score
   ↓
Count drivers + assign Risk_Data_Quality
   ↓
Require at least 2 drivers for dashboard publication
   ↓
Rank positive weighted contributions
   ↓
Top_Driver_1 / 2 / 3
   ↓
Dashboard output
   ↓
Backend handoff CSV
```

# 27. Conclusion

The final MVP risk engine was selected because it is:

- transparent,
- explainable,
- compatible with incomplete price data,
- easy to audit,
- and better aligned with the demonstrated strength of the available dataset
  than the tested ML alternatives.

The machine-learning experiments remain valuable as evidence of model
evaluation, but they are not presented as stronger than the measured results
support.